In [1]:
import pandas as pd
from pathlib import Path

In [2]:
raw_path = Path("../data/raw")

csv_files = list(raw_path.glob("*.csv"))

len(csv_files), csv_files

(11,
 [WindowsPath('../data/raw/dailyActivity_merged.csv'),
  WindowsPath('../data/raw/heartrate_seconds_merged.csv'),
  WindowsPath('../data/raw/hourlyCalories_merged.csv'),
  WindowsPath('../data/raw/hourlyIntensities_merged.csv'),
  WindowsPath('../data/raw/hourlySteps_merged.csv'),
  WindowsPath('../data/raw/minuteCaloriesNarrow_merged.csv'),
  WindowsPath('../data/raw/minuteIntensitiesNarrow_merged.csv'),
  WindowsPath('../data/raw/minuteMETsNarrow_merged.csv'),
  WindowsPath('../data/raw/minuteSleep_merged.csv'),
  WindowsPath('../data/raw/minuteStepsNarrow_merged.csv'),
  WindowsPath('../data/raw/weightLogInfo_merged.csv')])

In [ ]:
for file in csv_files:
    temp = pd.read_csv(file)
    print(f"{file.name}:{temp.shape}")

In [4]:
# data quality audit
quality_report = []

for file in csv_files:
    temp = pd.read_csv(file)

    quality_report.append({
        "file": file.name,
        "rows": temp.shape[0],
        "columns": temp.shape[1],
        "missing_values": temp.isna().sum().sum(),
        "duplicate_rows": temp.duplicated().sum()
    })

quality_df = pd.DataFrame(quality_report)

quality_df

,file,rows,columns,missing_values,duplicate_rows
0,dailyActivity_merged.csv,457,15,0,0
1,heartrate_seconds_merged.csv,1154681,3,0,0
2,hourlyCalories_merged.csv,24084,3,0,0
3,hourlyIntensities_merged.csv,24084,4,0,0
4,hourlySteps_merged.csv,24084,3,0,0
5,minuteCaloriesNarrow_merged.csv,1445040,3,0,0
6,minuteIntensitiesNarrow_merged.csv,1445040,3,0,0
7,minuteMETsNarrow_merged.csv,1445040,3,0,0
8,minuteSleep_merged.csv,198559,4,0,525
9,minuteStepsNarrow_merged.csv,1445040,3,0,0


In [5]:
weight_df = pd.read_csv("../data/raw/weightLogInfo_merged.csv")

weight_df.isna().sum()

Id                 0
Date               0
WeightKg           0
WeightPounds       0
Fat               31
BMI                0
IsManualReport     0
LogId              0
dtype: int64

In [6]:
sleep_df = pd.read_csv("../data/raw/minuteSleep_merged.csv")
duplicates = sleep_df[sleep_df.duplicated(keep=False)]

duplicates.head(10)

,Id,date,value,logId
79892,4319703577,4/5/2016 10:50:00 PM,3,11344563687
79893,4319703577,4/5/2016 10:51:00 PM,3,11344563687
79894,4319703577,4/5/2016 10:52:00 PM,2,11344563687
79895,4319703577,4/5/2016 10:53:00 PM,2,11344563687
79896,4319703577,4/5/2016 10:54:00 PM,2,11344563687
79897,4319703577,4/5/2016 10:55:00 PM,1,11344563687
79898,4319703577,4/5/2016 10:56:00 PM,1,11344563687
79899,4319703577,4/5/2016 10:57:00 PM,2,11344563687
79900,4319703577,4/5/2016 10:58:00 PM,2,11344563687
79901,4319703577,4/5/2016 10:59:00 PM,1,11344563687


In [9]:
duplicates.groupby(
    ["Id", "date", "value", "logId"]
).size().sort_values(ascending=False).head(10)

Id          date                  value  logId      
4319703577  4/5/2016 10:50:00 PM  3      11344563687    2
            4/5/2016 10:51:00 PM  3      11344563687    2
            4/5/2016 10:52:00 PM  2      11344563687    2
            4/5/2016 10:53:00 PM  2      11344563687    2
            4/5/2016 10:54:00 PM  2      11344563687    2
            4/5/2016 10:55:00 PM  1      11344563687    2
            4/5/2016 10:56:00 PM  1      11344563687    2
            4/5/2016 10:57:00 PM  2      11344563687    2
            4/5/2016 10:58:00 PM  2      11344563687    2
            4/5/2016 10:59:00 PM  1      11344563687    2
dtype: int64

In [10]:
sleep_clean = sleep_df.drop_duplicates().copy()
sleep_df.shape, sleep_clean.shape

((198559, 4), (198034, 4))

In [11]:
sleep_clean.duplicated().sum()

np.int64(0)

In [13]:
sleep_clean.dtypes
weight_df.dtypes

Id                  int64
Date               object
WeightKg          float64
WeightPounds      float64
Fat               float64
BMI               float64
IsManualReport       bool
LogId               int64
dtype: object

In [14]:
sleep_clean["date"] = pd.to_datetime(sleep_clean["date"])

In [15]:
weight_df["Date"] = pd.to_datetime(weight_df["Date"])

C:\Users\mukul\AppData\Local\Temp\ipykernel_16128\3946706220.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  weight_df["Date"] = pd.to_datetime(weight_df["Date"])


In [16]:
sleep_clean.dtypes

Id                int64
date     datetime64[ns]
value             int64
logId             int64
dtype: object

In [17]:
weight_df.dtypes

Id                         int64
Date              datetime64[ns]
WeightKg                 float64
WeightPounds             float64
Fat                      float64
BMI                      float64
IsManualReport              bool
LogId                      int64
dtype: object

In [18]:
# 1. Verify missing values
print("Weight missing values:")
print(weight_df.isna().sum())

print("\nSleep missing values:")
print(sleep_clean.isna().sum())

# 2. Verify duplicates
print("\nSleep duplicates:", sleep_clean.duplicated().sum())

# 3. Sort longitudinal data chronologically
sleep_clean = sleep_clean.sort_values(["Id", "date"]).reset_index(drop=True)
weight_df = weight_df.sort_values(["Id", "Date"]).reset_index(drop=True)

# 4. Create processed-data directory
from pathlib import Path

processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

# 5. Save cleaned datasets
sleep_clean.to_csv(
    processed_path / "minuteSleep_clean.csv",
    index=False
)

weight_df.to_csv(
    processed_path / "weightLogInfo_clean.csv",
    index=False
)

print("\nCleaned datasets saved successfully.")

Weight missing values:
Id                 0
Date               0
WeightKg           0
WeightPounds       0
Fat               31
BMI                0
IsManualReport     0
LogId              0
dtype: int64

Sleep missing values:
Id       0
date     0
value    0
logId    0
dtype: int64

Sleep duplicates: 0

Cleaned datasets saved successfully.


In [19]:
print("Sleep shape:", sleep_clean.shape)
print("Sleep duplicates:", sleep_clean.duplicated().sum())
print("Sleep date type:", sleep_clean["date"].dtype)

print("\nWeight shape:", weight_df.shape)
print("Weight date type:", weight_df["Date"].dtype)
print("Weight missing values:", weight_df.isna().sum().sum())

Sleep shape: (198034, 4)
Sleep duplicates: 0
Sleep date type: datetime64[ns]

Weight shape: (33, 8)
Weight date type: datetime64[ns]
Weight missing values: 31
